# IMPORT LIBRARIES

In [1]:
%load_ext autoreload
%autoreload 2

In [13]:
import os
import sys
import shutil
import datetime
import pandas as pd
from dateutil import relativedelta  
# from Config.paths import base_dir
base_dir = r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\DOSSIERS_UTILISATEURS\Yohan\02_BACKTEST_STRAT_IA\Backtest"
from Codes.OTHER_FUNC import *



secto_reco_path = r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_PTF_BLOOM\reco_secto_facto.xlsx"
list_noire_path = r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_BASE\_ ESG DATA\Liste_Noire_Exclusion.xlsx"
path_screen = r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_SCREEN_AGG\screen_aggregate.parquet"
path_returns = r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_RETURNS\returns.parquet"
path_ciq = r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_SCREEN_AGG\last_screenCIQ.parquet"

regional_ptfs_output_dir=r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_PTF_BLOOM\PTF"

### MSCI EMERGING

In [ ]:
from Codes.ML_PIPELINE import ML_MonthlyProdPipeline

import Config.config_EM as config
predictor_emerging = ML_MonthlyProdPipeline(  
    config.CONFIG,  
    mode="backtest",
    preprocessing=True,
    update_score_ML=True,
    allow_multiprocessing=True,
    output_path=os.path.join(base_dir, "Output_SCORE_SHAP"),
    output_file="SCORE_ML_EM_backtest"
)  

predictor_emerging.run()

In [ ]:
predictor_emerging.update_screen_aggregate() 

Backup created at: \\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_SCREEN_AGG\backup_screen\screen_aggregate_20260331_141356_before_ml.parquet


A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.




Updated 'Score ML' for 189850 rows.
Screen aggregate file updated successfully at: \\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_SCREEN_AGG\screen_aggregate.parquet


True

In [3]:
screen_agg_with_score = pd.read_parquet(r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_SCREEN_AGG\screen_aggregate.parquet")
msci_em = screen_agg_with_score[screen_agg_with_score["Weight in MSCI EM"] > 0]

### EMERGING

In [9]:
# Calcul du pourcentage de NaN par date
nan_per_date = msci_em.groupby("Date")["ESG_ANALYST_SCORE"].apply(lambda x: x.isna().mean() * 100)

print(nan_per_date)

Date
2004-12-31    100.000000
2005-01-31    100.000000
2005-02-28    100.000000
2005-03-31    100.000000
2005-04-29    100.000000
                 ...    
2025-11-28     43.060201
2025-12-31     43.274854
2026-01-30     43.311037
2026-02-27     44.398340
2026-03-31     44.647303
Name: ESG_ANALYST_SCORE, Length: 256, dtype: float64


In [ ]:
from datetime import datetime

bench = 'MSCI EM' #'STOXX EUROPE 600 dans MSCI World'
metrics = 'Score ML'
percentile = 0.20 # Top 600 * 0.10 = 60 boites
esg_exclusion = 0.2
ptf_name = 'ML EM Q1'
ponderation = 'Market cap'
top_mandatory = 2
cap_weight_threshold = 0.08
cut_mkt_cap = 2000

# screen_agg_with_score = pd.read_parquet(r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\DOSSIERS_UTILISATEURS\Yohan\02_BACKTEST_STRAT_IA\Backtest\Output_SCORE_SHAP_EM\SCORE_ML_EM_backtest.parquet")

builder_EM = PtfBuilder(msci_em, 
                    returns, 
                    bench=bench,
                    ptf_name=ptf_name, 
                    percentile=percentile,
                    # liste_noire=list_noire_path,
                    liste_noire=None,
                    metrics=metrics, 
                    top_mandatory=top_mandatory,
                    # cap_weight_threshold = cap_weight_threshold,
                    ponderation = ponderation,
                    esg_exclusion=None,
                    Top =False,
                    )

# builder_EM.generic_histo_seclist(start_date=datetime.datetime(2010,2,1),freq_rebal=1,  cluster = 'bucket') 

builder_EM.generic_histo_seclist(start_date=datetime(2010,7,1),freq_rebal=1) 

builder_EM.backtest_plot_ptf_bench(save_path=False)

In [ ]:
from datetime import datetime

bench = 'MSCI EM' #'STOXX EUROPE 600 dans MSCI World'
metrics = 'Score ML'
percentile = 0.20 # Top 600 * 0.10 = 60 boites
esg_exclusion = 0.2
ptf_name = 'ML EM Q1'
ponderation = 'Market cap'
top_mandatory = 2
cap_weight_threshold = 0.08
cut_mkt_cap = 2000

# screen_agg_with_score = pd.read_parquet(r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\DOSSIERS_UTILISATEURS\Yohan\02_BACKTEST_STRAT_IA\Backtest\Output_SCORE_SHAP_EM\SCORE_ML_EM_backtest.parquet")

builder_EM = PtfBuilder(msci_em, 
                    returns, 
                    bench=bench,
                    ptf_name=ptf_name, 
                    percentile=percentile,
                    # liste_noire=list_noire_path,
                    liste_noire=None,
                    metrics=metrics, 
                    top_mandatory=top_mandatory,
                    # cap_weight_threshold = cap_weight_threshold,
                    ponderation = ponderation,
                    esg_exclusion=None,
                    Top =True,
                    )

# builder_EM.generic_histo_seclist(start_date=datetime.datetime(2010,2,1),freq_rebal=1,  cluster = 'bucket') 

builder_EM.generic_histo_seclist(start_date=datetime(2010,7,1),freq_rebal=1) 

builder_EM.backtest_plot_ptf_bench(save_path=False)

In [39]:
(msci_em_last.groupby('Exchange Country Name')['Weight in MSCI EM'].sum() / msci_em_last['Weight in MSCI EM'].sum()).sort_values(ascending =False)

Exchange Country Name
HONG KONG               0.216641
SOUTH KOREA             0.150365
TAIWAN                  0.116794
UNITED STATES           0.085761
INDIA                   0.083513
BRAZIL                  0.075415
SOUTH AFRICA            0.060121
MEXICO                  0.034741
RUSSIA                  0.023899
INDONESIA               0.022786
MALAYSIA                0.022695
THAILAND                0.020926
POLAND                  0.013171
CHILE                   0.012729
PHILIPPINES             0.011356
TURKEY                  0.011176
UNITED ARAB EMIRATES    0.007279
UNITED KINGDOM          0.006744
QATAR                   0.005721
COLOMBIA                0.004481
HUNGARY                 0.003346
GREECE                  0.003118
GERMANY                 0.002668
CZECH REPUBLIC          0.001830
EGYPT                   0.001231
PAKISTAN                0.000948
CHINA                   0.000545
Name: Weight in MSCI EM, dtype: float64

In [42]:
weight_pays_ptf

Country Group
HONG KONG      0.227473
Others         0.492011
SOUTH KOREA    0.157883
TAIWAN         0.122634
Name: Weight, dtype: float64

# fonction

In [ ]:
def adjust_bench_weight_with_recommandation( df, reco_secto  , date):
    """
    1 - Calculer les weights sectoriels du bench 
    2 - Appliquer les reco sectorielles si besoin
    """
    weight_neutral = "ICB 19"
    if weight_neutral == "ICB 19":
        weight_secto_bench = \
                df.groupby(' Benchmark ICB Supersector ')['Weight in ' + bench].sum() / df['Weight in ' + bench].sum()
        icb_missing = set([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]) - set(df[' Benchmark ICB Supersector '].unique())
        if len(icb_missing) > 0:
            print(f"Warning: follwing sectors are missing in benchmark: {list(icb_missing)}")

            indices_to_delete = [int(icb)-1 for icb in icb_missing] # find out where need to be deleted and then mark it as 1
            reco_secto = np.delete(np.array(reco_secto), indices_to_delete) # delete function can delete several values at a same time, no need to do iteration


        # Recommandation sectorielle
        weight_secto_bench = weight_secto_bench + (np.array(reco_secto)*1)   

        # Adjust small weight sectors
        small_weight_mask = weight_secto_bench < 0.0025
        sectors_to_be_adjusted = weight_secto_bench[small_weight_mask].index
        if not sectors_to_be_adjusted.empty:
            # print(f"Warning: follwing sectors have weight lower than 0.0025, their weight will be replace as 0.0025: {sectors_to_be_adjusted.tolist()}")
            weight_secto_bench[small_weight_mask] = 0.0025

    
    return weight_secto_bench


def neutralise_score_by_secteur(df, list_score_col):
    """
    Neutraliser sectoriellement le score pour piocher les tops par secteur par la suite
    """
    df = df.copy()
    df.loc[:, list_score_col] = df[list_score_col].rank(pct=True)
    df.loc[:, list_score_col] = (df[list_score_col] - df[list_score_col].min())/(df[list_score_col].max() - df[list_score_col].min()) # min max scaler
    score_neutral = "ICB 19"
 
    if score_neutral == "ICB 19":
        for secto in df[' Benchmark ICB Supersector '].unique():
            df.loc[df[' Benchmark ICB Supersector '] == secto, list_score_col] = df.loc[df[' Benchmark ICB Supersector '] == secto, list_score_col].rank(pct=True)
            df.loc[df[' Benchmark ICB Supersector '] == secto, list_score_col] = (df.loc[df[' Benchmark ICB Supersector '] == secto, list_score_col] - df.loc[df[' Benchmark ICB Supersector '] == secto, list_score_col].min())/(df.loc[df[' Benchmark ICB Supersector '] == secto, list_score_col].max() - df.loc[df[' Benchmark ICB Supersector '] == secto, list_score_col].min())
    return df

def select_titles(group, max_weight_threshold, column):
    """
        Sélectionne un nombre minimum de titres dans un secteur afin de respecter
        une contrainte de poids maximum par titre.
    
        Cette fonction calcule le poids total du secteur, détermine le nombre
        minimal de titres nécessaires pour que chacun ne dépasse pas le seuil
        de poids maximum défini, puis sélectionne les titres ayant les valeurs
        les plus élevées en mérique choisie (ex. Score ML).
    
        Paramètres
        ----------
        group : pandas.DataFrame
            Sous-ensemble du DataFrame contenant uniquement les titres du secteur
            considéré. Doit contenir au minimum la colonne :
            - "Weight in <benchmark>" : poids de chaque titre dans le benchmark.
    
        max_weight_threshold : float
            Poids maximum toléré pour un titre dans le portefeuille. Le nombre
            minimal de titres est calculé de manière à ce qu'aucun ne dépasse ce seuil.
    
        column : str
            Nom de la colonne utilisée pour sélectionner les titres avec les valeurs
            les plus élevées (utilisée avec `nlargest`).
    
        Retour
        ------
        pandas.DataFrame
            Un DataFrame contenant uniquement les titres sélectionnés selon la
            contrainte de poids et la logique de ranking sur la colonne donnée.
    
        Notes
        -----
        - Le nombre minimal de titres est arrondi à l’entier supérieur.
        - Si le secteur a un poids total de 0, aucun titre ne sera sélectionné.
    
    """
    sector_weight = group['Weight in ' + bench].sum()  # Get the sector's total weight
    min_titles_needed = (sector_weight // max_weight_threshold) + (1 if sector_weight % max_weight_threshold != 0 else 0) # Division euclidienne pour connaitre le min de titre a avoir
    
    sector = group[" Benchmark ICB Supersector "].unique()

    # Choisir les minimum de titres pour respecter la contrainte
    selected_titles = group.nlargest(int(min_titles_needed), column)  
    
    return selected_titles

In [70]:

"""
Generate Best Scored Sec List for 1 Month, According to the Metrics Chosen
"""

if isinstance(screen_agg_monthly, pd.DataFrame):
    screen=copy.deepcopy(screen_agg_monthly)
elif screen_agg_monthly==None: # If single month dataframe is not defined, then use the last month data to generate ptf
    screen = screen[screen['Date'] == screen['Date'].max()] 


if type(metrics)==str:
    list_score_col = [metrics]
else:
    list_score_col = metrics


################ use only the last month's screen (production mode) ################ 
date = pd.to_datetime(screen['Date'].max())
# screen=screen[screen['Date']==date]

if screen.index.duplicated().any():
    screen = screen[~screen.index.duplicated(keep='first')]

################ Merging les tickers secondaires ################
screen = merge_ticker_secondaire(screen, bench)

################ Filtrage Bench ################
df = screen[screen['Weight in ' + bench]>0] # on conserve que les weights positifs
list_exclusion_bench = screen.loc[~screen.index.isin(df.index)].index.to_list() # For later merge with other list of exclusion

################ Fixer le nombre de boite à choisir pour plus tard avant que le df soit modifié ################
if percentile > 1:   ##### If percentile is bigger than 1, then this variable means exact number of securities to pick
    nb_securities=percentile
else: ##### If percentile is less than 1, then this variable means the percentage of securities in investable univers to pick
    nb_securities = round(len(df) * percentile)

################ donne le 1er jour du mois suivant ################
date +=pd.offsets.MonthBegin(1)

################################################################################################################################

################################################################################################################################
# regression de Benchmark value sur weight in bench pour compléter les valeurs manquantes
fit = np.polyfit(df.loc[pd.isna(df['Benchmark Market Value Millions in EUR ']) == False, 'Weight in ' + bench],df.loc[pd.isna(df['Benchmark Market Value Millions in EUR ']) == False, 'Benchmark Market Value Millions in EUR '], deg = 1)
func = np.poly1d(fit)
df.loc[pd.isna(df['Benchmark Market Value Millions in EUR ']),'Benchmark Market Value Millions in EUR '] = func(df.loc[pd.isna(df['Benchmark Market Value Millions in EUR ']),'Weight in ' + bench])
# Market cut filtrage
df.loc[df['Benchmark Market Value Millions in EUR '] <= cut_mkt_cap, list_score_col] = np.NaN
list_exclusion_market_cut = df[df['Benchmark Market Value Millions in EUR '] <= cut_mkt_cap].index.to_list()  # on note les entreprises hors de benchmark, For later merge with other list of exclusion

df = df.copy()
df.loc[:, 'Date'] = date

# Ajuster le poids des titres dans l'indice (smoothing)
# df=adjust_companies_ponderation(df)

# Recommandtions Sectorielles à la date donnée

reco_secto = [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]
if isinstance(reco_secto, list):
    reco_secto =copy.deepcopy(reco_secto)
elif isinstance(reco_secto, pd.DataFrame):
    try:
        reco_secto = reco_secto.loc[date].to_list() 
    except : 
        print(f"{date} not in reco_secto")
        raise KeyError

# Appilquer les reco sectorielle aux secteurs
df = add_country_group(df)
weight_secto_bench = adjust_bench_weight_with_recommandation(df, reco_secto, date)
weight_pays_bench = df.groupby('Country Group')['Weight in ' + bench].sum() / df['Weight in ' + bench].sum()

# Initiate Dataframe for liste exclusion
titles_excluded = pd.DataFrame(columns=['Date', "Raison Exclusion"])

# Filtrage ESG only if we choose Top ptf
Top = True
# if Top:
#     # if score_pivot_esg is string then it will find corresponding item note in excel, 
#     # if score_pivot_esg is float, it will use it as threshold, 
#     # if score_pivot_esg is None, filter by esg pivot score will not apply
#     if isinstance(score_pivot_esg, str):
#         print(f"Récupération de Score Pivot ESG avec index {score_pivot_esg}......")
#         score_pivot_esg = get_esg_pivot_score(bench_name_in_excel = score_pivot_esg) 
#     if isinstance(score_pivot_esg, float):
#         print(f"Score Pivot ESG est {score_pivot_esg}")
#     df, titles_excluded = filtrage_esg_liste_noire(df,date)  

# Combine titles_excluded with Market Cut Exclusion
default_date = titles_excluded['Date'].iloc[0] if not titles_excluded.empty else date  # Add date for exclusion liste dataframe

# Create exclusion dataframe for market cut reason
new_entries_exclusion = pd.DataFrame({
    'Date': [default_date] * len(list_exclusion_market_cut),
    'Raison Exclusion': ['Cut Market'] * len(list_exclusion_market_cut)
}, index=list_exclusion_market_cut)

titles_excluded = pd.concat([titles_excluded, new_entries_exclusion], axis=0) # Concat avec le dataframe de début

# Create exclusion dataframe for not in bench reason reason
new_entries_not_in_bench = pd.DataFrame({
    'Date': [default_date] * len(list_exclusion_bench),
    'Raison Exclusion': ["Not in Bnech"] * len(list_exclusion_bench)
}, index=list_exclusion_bench)


if not new_entries_not_in_bench.empty and not new_entries_not_in_bench.isna().all().all():
    titles_excluded = pd.concat([titles_excluded, new_entries_not_in_bench], axis=0)
    

df = neutralise_score_by_secteur(df, list_score_col) 

df["Raison Repechage"] = ""

columns = ['PTF', 'ISIN', 'Weight', 'Date', "Raison Repechage"]
result_sec_list = pd.DataFrame()

Exchange Country Name
TAIWAN         0.226900
HONG KONG      0.199042
SOUTH KOREA    0.165182
Name: Weight in MSCI EM, dtype: float64


C:\Users\RADETYO\AppData\Local\Temp\ipykernel_10008\924606555.py:99: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



In [29]:
weight_neutral = "ICB 19"

In [72]:


for i in range(len(list_score_col)):

    if Top == True:
        df_top = df.nlargest(nb_securities,list_score_col[i])
        df_top['Raison Repechage'] = list_score_col[i]  # Mettre la métrique de repechage comme raison, ex. Score ML/ Growth Avg Percentile

        # Selection du minimum de titres minimum par secteur pour respecter la contrainte de poid max (cap_weight_threshold)
        if cap_weight_threshold != None:
            df_top_sector =  df.groupby(' Benchmark ICB Supersector ').apply(select_titles, max_weight_threshold=cap_weight_threshold, column = list_score_col[i])
            df_top_sector = df_top_sector.drop( columns = [" Benchmark ICB Supersector "] )
            df_top_sector = df_top_sector.reset_index(drop=False)
            df_top_sector.index = df_top_sector["ISIN"]
            df_top_sector = df_top_sector.drop( columns = ["ISIN"] )
            df_top_sector["Raison Repechage"] = "Sector"
        
            # Concat les deux top list
            df_top_combined = pd.concat([df_top, df_top_sector], axis=0)
            df_top = df_top_combined[~df_top_combined.index.duplicated(keep='first')]  # Prioritize top selected with classical way

        # Add non selected titles in list exclusion beacause of metrics
        list_exclusion_metrics = df.loc[~df.index.isin(df_top.index)].index.to_list()
        new_entries_exclusion = pd.DataFrame({
            'Date': [default_date] * len(list_exclusion_metrics),
            'Raison Exclusion': [f"Bad {list_score_col[i]}"] * len(list_exclusion_metrics)
        }, index=list_exclusion_metrics)

        # Append to the existing final_df
        titles_excluded = pd.concat([titles_excluded, new_entries_exclusion], axis=0)
        titles_excluded.index.name = "ISIN"
        if "ISIN" in titles_excluded.columns:
            titles_excluded = titles_excluded.drop( columns = ["ISIN"] )
        titles_excluded = titles_excluded.reset_index()


    if Top == False:
        df_top=df.nsmallest(nb_securities,list_score_col[i])
        df_top['Raison Repechage'] = "Worst Metric"

    if isinstance(top_mandatory, int) or isinstance(top_mandatory, float):
        nb_top_mandatory = int(top_mandatory)
        # print(f"Top Mandatory is activated, top {nb_top_mandatory} companies in bench will be added to sec list")
        liste_top_mandatory = df.nlargest(nb_top_mandatory, 'Weight in ' + bench)
        liste_top_mandatory['Raison Repechage'] = "Top Obligatoire par Région"

        df_top_combined = pd.concat([liste_top_mandatory, df_top], axis=0)
        df_top = df_top_combined[~df_top_combined.index.duplicated(keep='first')]  # Prioritize top selected with top mandatory


    ##### Ajustement Secteur Neutre
    temp_df = pd.DataFrame(columns = columns)
    temp_df['ISIN'] = df_top.index
    if weight_neutral == "ICB 19":
        temp_df['Secto'] = df_top[' Benchmark ICB Supersector '].values
    elif weight_neutral == "ICB 11":
        temp_df['Secto'] = df_top[' Benchmark ICB Industry '].values
    temp_df['Weight'] = df_top['Benchmark Market Value Millions in EUR '].values

    temp_df['Score'] = df_top[list_score_col[i]].values
    temp_df['Date'] = df_top['Date'].values
    temp_df['Raison Repechage'] = df_top['Raison Repechage'].values
    temp_df["Country Group"] = df_top["Country Group"].values

    ###################### Security check : secto in temp_df is a subset of secto in weight_secto_bench ############################
    temp_df_sectors = set(temp_df['Secto'].unique())
    benchmark_sectors = set(weight_secto_bench.index)
    if not temp_df_sectors.issubset(benchmark_sectors):
        missing_sectors = temp_df_sectors.difference(benchmark_sectors)
        raise ValueError(f"Error: Sectors in temp_df {missing_sectors} are not defined in weight_secto_bench")
    #################################################################################################################################

    if weight_neutral != None:
        secto_weight_sum = temp_df.groupby('Secto')['Weight'].transform('sum')
        secto_benchmark_weight = temp_df['Secto'].map(weight_secto_bench)
        scaling_factor = secto_benchmark_weight / secto_weight_sum
        temp_df['Weight'] = temp_df['Weight'] * scaling_factor
        temp_df['Weight'] = temp_df['Weight'] / temp_df['Weight'].sum()

    ############ cap weight by sector if necessary ################
    # if cap_weight_threshold != None:
    #     # print(f"Capping generated portfolio, no more than {cap_weight_threshold} for each title")
    #     temp_df = cap_weight_by_sector(temp_df)

    ##### Give name to ptf generated
    temp_df['PTF'] = "name"

    result_sec_list = pd.concat([result_sec_list,temp_df], ignore_index=True)


sec_list_monthly = result_sec_list.copy(deep=True)
# print(f"Monthly sec list is generated for {date}, you can check 'sec_list_monthly' attribute for more details.")

list_exclusion_monthly = titles_excluded.copy(deep=True)



C:\Users\RADETYO\AppData\Local\Temp\ipykernel_10008\2748086592.py:9: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [112]:
weight_secto_bench = df.groupby(' Benchmark ICB Supersector ')['Weight in ' + bench].sum() / df['Weight in ' + bench].sum()
weight_pays_bench = df.groupby('Country Group')['Weight in ' + bench].sum() / df['Weight in ' + bench].sum()

weight_secto_bench.index.name = "Secto"

In [92]:
import cvxpy as cp
import numpy as np
import pandas as pd

def optimize_weights(df, target_secto, target_country):
    n = len(df)
    w = cp.Variable(n)

    w0 = df["Weight"].values

    constraints = []

    # Budget
    constraints.append(cp.sum(w) == 1)

    # Bornes (optionnel mais conseillé)
    constraints.append(w >= 0)
    # constraints.append(w <= 0.05)  # exemple

    # Secteurs
    for s, target in target_secto.items():
        print(s, target)
        idx = df["Secto"] == s
        constraints.append(cp.sum(w[idx]) == target)

    # Pays
    for c, target in target_country.items():
        print(c, target)
        idx = df["Country Group"] == c
        constraints.append(cp.sum(w[idx]) == target)

    # Objectif
    objective = cp.Minimize(cp.sum_squares(w - w0))

    prob = cp.Problem(objective, constraints)
    prob.solve()
    print(prob.value)

    df["Weight_opt"] = w.value
    return df

In [147]:
# optimize_weights.py
import cvxpy as cp
import numpy as np
import pandas as pd

def optimize_weights(df, target_secto, target_country):
    n = len(df)
    w = cp.Variable(n)
    w0 = df["Weight"].values
    constraints = []

    # Budget
    constraints.append(cp.sum(w) == 1)
    # Bornes (optionnel mais conseillé)
    constraints.append(w >= 0)
    
    # Secteurs
    for s, target in target_secto.items():
        idx = df["Secto"] == s
        constraints.append(cp.sum(w[idx]) == target)
    # Pays
    for c, target in target_country.items():
        idx = df["Country Group"] == c
        constraints.append(cp.sum(w[idx]) == target)

    # Objectif
    objective = cp.Minimize(cp.sum_squares(w - w0))
    prob = cp.Problem(objective, constraints)
    prob.solve()

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"Optimisation non convergente : status={prob.status}")

    df["Weight_opt"] = w.value
    return df

    
def check_solution(df, w_opt, target_secto, target_country, tolerance=1e-6):
    """Vérifie la solution retournée par l’optimiseur."""
    # Somme totale
    assert abs(np.sum(w_opt) - 1) <= tolerance, "La somme des poids n’est pas égale à 1."

    # Non‑négativité
    assert np.all(w_opt >= -tolerance), "Certains poids sont négatifs."

    # Secteurs
    for s, target in target_secto.items():
        idx = df["Secto"] == s
        assert abs(np.sum(w_opt[idx]) - target) <= tolerance, f"Secteur {s} hors cible."

    # Pays
    for c, target in target_country.items():
        idx = df["Country Group"] == c
        assert abs(np.sum(w_opt[idx]) - target) <= tolerance, f"Pays {c} hors cible."

    # (Optionnel) Vérifier que le coût est proche de `prob.value`
    # → à intégrer dans le test autonome
    print("✅ Vérification terminée : solution conforme aux contraintes.")

In [94]:
# optimize_weights.py
import cvxpy as cp
import numpy as np
import pandas as pd

# … (votre fonction optimize_weights reste inchangée) …

def is_feasible(df, target_secto, target_country, solver=None):
    """
    Retourne True si un point satisfaisant toutes les contraintes existe.
    """
    n = len(df)
    w = cp.Variable(n)
    constraints = []

    # Budget
    constraints.append(cp.sum(w) == 1)
    # Bornes
    constraints.append(w >= 0)
    # Secteurs
    for s, target in target_secto.items():
        idx = df["Secto"] == s
        constraints.append(cp.sum(w[idx]) == target)
    # Pays
    for c, target in target_country.items():
        idx = df["Country Group"] == c
        constraints.append(cp.sum(w[idx]) == target)

    # Objectif nul – on ne cherche qu’un point valide
    prob = cp.Problem(cp.Minimize(0), constraints)
    prob.solve(solver=solver)

    # CVXPY considère comme faisable si le problème est optimal ou proche
    return prob.status in {"optimal", "optimal_inaccurate", "optimal_feasible"}

In [ ]:
# optimize_weights.py
import cvxpy as cp
import numpy as np
import pandas as pd

# … (votre fonction optimize_weights reste inchangée) …

def is_feasible(df, target_secto, target_country,
                enforce_nonnegative=True, enforce_budget=True, solver=None):
    """
    Vérifie si un point satisfaisant toutes les contraintes existe.
    Arguments facultatifs :
        enforce_nonnegative : désactive la contrainte w >= 0
        enforce_budget      : désactive la contrainte sum(w) == 1
    """
    # Vérification préalable des sommes des cibles (budget)
    if enforce_budget:
        total_secto = sum(target_secto.values)
        total_country = sum(target_country.values)
        if abs(total_secto - 1) > 1e-6 or abs(total_country - 1) > 1e-6:
            # Diagnostic utile
            print(f"[DEBUG] Somme cible secteur={total_secto:.4f}, pays={total_country:.4f} → >1 ou <1")
            return False

    n = len(df)
    w = cp.Variable(n)
    constraints = []

    # Budget
    if enforce_budget:
        constraints.append(cp.sum(w) == 1)

    # Bornes
    if enforce_nonnegative:
        constraints.append(w >= 0)

    # Secteurs
    for s, target in target_secto.items():
        idx = df["Secto"] == s
        constraints.append(cp.sum(w[idx]) == target)

    # Pays
    for c, target in target_country.items():
        idx = df["Country Group"] == c
        constraints.append(cp.sum(w[idx]) == target)

    # Objectif nul – on ne cherche qu’un point valide
    prob = cp.Problem(cp.Minimize(0), constraints)
    prob.solve(solver=solver)

    return prob.status in {"optimal", "optimal_inaccurate", "optimal_feasible"}

In [102]:
sum(weight_secto_bench.values)

1.0008721171435742

In [118]:
print("Faisable (budget & non‑neg) :", is_feasible(temp_df, weight_secto_bench, weight_pays_bench))
print("Faisable (budget désactivé) :", is_feasible(temp_df, weight_secto_bench, weight_pays_bench, enforce_budget=False))
print("Faisable (non‑neg désactivé) :", is_feasible(temp_df, weight_secto_bench, weight_pays_bench, enforce_nonnegative=False))

Faisable (budget & non‑neg) : True
Faisable (budget désactivé) : True
Faisable (non‑neg désactivé) : True


In [119]:
is_feasible(temp_df, weight_secto_bench, weight_pays_bench)

True

In [214]:
len(temp_df["Secto"].unique())

19

In [210]:
df_result = optimize_weights(df_pb, weight_secto_bench, weight_pays_bench)

Nombre de NaN dans la colonne 'Score' : 0
Nombre de NaN dans la colonne 'Score' : 0
Score pondÃ©rÃ© avant optimisation: 0.6363113761607464
Optim score IA
RatÃ©â€¯!â€¯:  Optimisation non convergente : status=infeasible


In [176]:
df_result = optimize_weights(temp_df, weight_secto_bench, weight_pays_bench)

Score pondÃ©rÃ© avant optimisation: 0.9947103184433106
Optim score IA
Score pondÃ©rÃ© aprÃ¨s optimisation: 0.9947103184433106


In [177]:
df_result

,PTF,ISIN,Weight,Date,Raison Repechage,Secto,Score,Country Group,Weight_opt
0,name,TW0002330008,-2.757767e-10,2026-04-01,Top Obligatoire par Région,16.0,0.693252,TAIWAN,0.152440
1,name,KR7005930003,-1.093399e-10,2026-04-01,Top Obligatoire par Région,17.0,0.903226,SOUTH KOREA,0.041428
2,name,KYG875721634,-2.984999e-10,2026-04-01,Top Obligatoire par Région,16.0,0.662577,HONG KONG,0.059358
3,name,KR7000660001,9.700999e-10,2026-04-01,Top Obligatoire par Région,16.0,0.981595,SOUTH KOREA,0.039442
4,name,KYG017191142,-3.377823e-10,2026-04-01,Top Obligatoire par Région,15.0,0.000000,HONG KONG,0.033854
...,...,...,...,...,...,...,...,...,...
239,name,HK0992009065,-2.477141e-10,2026-04-01,Score ML,16.0,0.828221,HONG KONG,0.000947
240,name,CNE100000L63,-2.203948e-10,2026-04-01,Score ML,9.0,0.827586,Others,0.002933
241,name,CNE100000627,-2.243100e-10,2026-04-01,Score ML,2.0,0.822695,Others,0.003003
242,name,TW0002474004,-2.028271e-10,2026-04-01,Score ML,16.0,0.822086,TAIWAN,0.000833


In [163]:
weight_secto_ptf = df_result.groupby('Secto')['Weight'].sum() / df_result['Weight'].sum()
weight_pays_ptf = df_result.groupby('Country Group')['Weight'].sum() / df_result['Weight'].sum()

In [179]:
df_result.sort_values("Weight", ascending = False)

,PTF,ISIN,Weight,Date,Raison Repechage,Secto,Score,Country Group,Weight_opt
84,name,TW0002317005,2.155546e-01,2026-04-01,Score ML,16.0,0.975460,TAIWAN,0.009199
55,name,KR7024110009,1.059120e-01,2026-04-01,Score ML,2.0,1.000000,SOUTH KOREA,0.001635
28,name,CNE1000003K3,5.671623e-02,2026-04-01,Score ML,3.0,1.000000,HONG KONG,0.001562
80,name,KYG5074A1004,4.543778e-02,2026-04-01,Score ML,15.0,1.000000,HONG KONG,0.002242
79,name,KR7005931001,4.221519e-02,2026-04-01,Score ML,17.0,1.000000,SOUTH KOREA,0.041428
...,...,...,...,...,...,...,...,...,...
237,name,CNE100001TQ9,-2.475406e-10,2026-04-01,Score ML,6.0,0.829268,HONG KONG,0.001939
239,name,HK0992009065,-2.477141e-10,2026-04-01,Score ML,16.0,0.828221,HONG KONG,0.000947
0,name,TW0002330008,-2.757767e-10,2026-04-01,Top Obligatoire par Région,16.0,0.693252,TAIWAN,0.152440
2,name,KYG875721634,-2.984999e-10,2026-04-01,Top Obligatoire par Région,16.0,0.662577,HONG KONG,0.059358


In [140]:
df.sort_values("Weight in MSCI EM", ascending = False)[["Weight in MSCI EM"]]

,Weight in MSCI EM
ISIN,
TW0002330008,1.343089e-01
KR7005930003,5.571287e-02
KYG875721634,3.804060e-02
KR7000660001,3.040405e-02
KYG017191142,2.531538e-02
...,...
CNE100004PM0,1.920082e-05
CNE100001KV8,1.799119e-05
CNE000000SV4,1.255459e-05


In [164]:
weight_secto_ptf

Secto
1.0     0.030818
2.0     0.159327
3.0     0.054008
4.0     0.011830
5.0     0.012209
6.0     0.023635
7.0     0.021199
8.0     0.030013
9.0     0.073416
10.0    0.027097
11.0    0.001564
12.0    0.044281
13.0    0.011148
14.0    0.011990
15.0    0.043220
16.0    0.312987
17.0    0.097950
18.0    0.011616
19.0    0.021692
Name: Weight, dtype: float64

In [165]:
df

,Date,Company SEDOL,Symbol,Name,Exchange Country Name,FactSet Ind,FactSet Economy,Curncy Iso,Exchange Country Region,Benchmark Country English,...,Weight in MSCI WORLD 2,Weight in Univ ML EU 2,Weight in Univ ML US 2,Weight in Univ ML OTHER 2,Perf5D,Perf1M,Perf3M,Perf6M,Country Group,Raison Repechage
ISIN,,,,,,,,,,,,,,,,,,,,,
CNE100003P74,2026-04-01,B0Z9G3-R,B0Z9G3-R,"Shenzhen Transsion Holding Co., Ltd. Class A",CHINA,Telecommunications Equipment,TECHNOLOGY,CNY,Asia,None,...,NaN,NaN,NaN,NaN,0.030077,0.017378,-0.184895,-0.303096,Others,
HK0992009065,2026-04-01,B14XHQ-R,B14XHQ-R,Lenovo Group Limited,HONG KONG,Computer Processing Hardware,TECHNOLOGY,HKD,Asia,Hong Kong,...,NaN,NaN,NaN,NaN,-0.006393,-0.023781,-0.019612,-0.184880,HONG KONG,
KYG5496K1242,2026-04-01,B2D16H-R,B2D16H-R,Li Ning Company Limited,HONG KONG,Apparel/Footwear,CONSUMER NON-DURABLES,HKD,Asia,China,...,NaN,NaN,NaN,NaN,-0.023985,0.028528,0.096922,0.285706,HONG KONG,
CNE000000V89,2026-04-01,B3MPYS-R,B3MPYS-R,"Shanghai International Airport Co., Ltd. Class A",CHINA,Other Transportation,INDUSTRIALS,CNY,Asia,None,...,NaN,NaN,NaN,NaN,0.002634,-0.074014,-0.140631,-0.084471,Others,
ID1000117609,2026-04-01,B3R2J2-R,B3R2J2-R,PT Bumi Resources Minerals Tbk Class A,INDONESIA,Precious Metals,BASIC MATERIALS,IDR,Asia,None,...,NaN,NaN,NaN,NaN,0.077827,-0.257914,-0.431624,-0.260922,Others,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CNE000001FJ4,2026-04-01,XQ8RN1-R,XQ8RN1-R,Zhejiang Longsheng Group Co. Ltd. Class A,CHINA,Chemicals: Specialty,INDUSTRIALS,CNY,Asia,None,...,NaN,NaN,NaN,NaN,0.023945,-0.179228,0.199104,0.342573,Others,
INE016A01026,2026-04-01,XQFR4Q-R,XQFR4Q-R,Dabur India Limited,INDIA,Household/Personal Care,CONSUMER NON-DURABLES,INR,Asia,India,...,NaN,NaN,NaN,NaN,-0.031724,-0.216730,-0.238850,-0.202033,Others,
PLBZ00000044,2026-04-01,XQG0NZ-R,XQG0NZ-R,Santander Bank Polska SA,POLAND,Regional Banks,FINANCE,PLN,East Europe,Poland,...,NaN,NaN,NaN,NaN,0.027250,0.047375,0.027824,0.200977,Others,


In [152]:
weight_secto_bench

Secto
1.0     0.030877
2.0     0.159125
3.0     0.054047
4.0     0.011886
5.0     0.012256
6.0     0.023690
7.0     0.021209
8.0     0.030044
9.0     0.073115
10.0    0.027160
11.0    0.001628
12.0    0.044342
13.0    0.011177
14.0    0.012044
15.0    0.043306
16.0    0.312830
17.0    0.097856
18.0    0.011675
19.0    0.021735
Name: Weight in MSCI EM, dtype: float64

In [166]:
weight_pays_ptf

Country Group
HONG KONG      0.199118
Others         0.408847
SOUTH KOREA    0.165080
TAIWAN         0.226955
Name: Weight, dtype: float64

In [167]:
weight_pays_bench

Country Group
HONG KONG      0.199042
Others         0.408876
SOUTH KOREA    0.165182
TAIWAN         0.226900
Name: Weight in MSCI EM, dtype: float64

In [53]:
import pandas as pd

def add_country_group(df,
                      weight_col='Weight in MSCI EM',
                      country_col='Exchange Country Name',
                      top_n=3,
                      new_col='Country Group'):
    """
    Ajoute une colonne indiquant les `top_n` pays les plus lourds
    (selon `weight_col`) et met "Others" pour les autres pays.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame contenant au moins `weight_col` et `country_col`.
    weight_col : str
        Nom de la colonne contenant les poids MSCI EM.
    country_col : str
        Nom de la colonne contenant les noms des pays.
    top_n : int
        Nombre de pays à retenir comme « top ».
    new_col : str
        Nom de la colonne à créer.
    
    Returns
    -------
    pd.DataFrame
        Le DataFrame d'origine avec la colonne `new_col` ajoutée.
    """
    # Somme des poids par pays
    top_countries = (
        df.groupby(country_col)[weight_col]
          .sum()
          .nlargest(top_n)
          .index
          .tolist()
    )
    print(df.groupby(country_col)[weight_col]
          .sum()
          .nlargest(top_n))
    
    # Crée la nouvelle colonne
    df[new_col] = df[country_col].where(df[country_col].isin(top_countries),
                                       other='Others')
    return df